# M2: Adding Sentiment Features to Earnings Prediction

M2 extends the M0/M1 baselines by adding pre-earnings **sentiment features** from the 7-day news window [ED-7, ED-1].

**New features:**
- `sent_mean` — average polarity of all articles in [ED-7, ED-1]
- `sent_trend` — sentiment momentum (later half vs earlier half of window)
- `sent_delta` — sentiment anomaly vs quiet period [ED-37, ED-30]
- `news_volume` — total article count (attention/hype proxy)

**Carried from M1:**
- `surprise_pct` — EPS surprise magnitude

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.size'] = 12

conn = sqlite3.connect('../data/hackathon.db')
print('DB connected')

## 1. Build Target + M1 Features

Reuse the same target construction as notebook 01.

In [ ]:
earnings = pd.read_sql('SELECT * FROM earnings', conn)
prices = pd.read_sql('SELECT * FROM daily_prices', conn)
prices['date'] = pd.to_datetime(prices['date'])
earnings['earnings_date'] = pd.to_datetime(earnings['earnings_date'])

records = []
for _, evt in earnings.iterrows():
    tk = evt['ticker']
    ed = evt['earnings_date']
    tk_prices = prices[prices['ticker'] == tk].sort_values('date')
    
    post = tk_prices[tk_prices['date'] > ed].head(5)
    if len(post) < 5:
        continue
    pre = tk_prices[tk_prices['date'] <= ed].tail(1)
    if pre.empty:
        continue
    
    p0 = pre['close'].values[0]
    p5 = post['close'].values[-1]
    ret_5d = (p5 - p0) / p0
    
    records.append({
        'ticker': tk,
        'earnings_date': ed,
        'surprise_pct': evt['surprise_pct'],
        'ret_5d': ret_5d,
        'target': 1 if ret_5d > 0 else 0,
    })

df = pd.DataFrame(records).sort_values('earnings_date').reset_index(drop=True)
print(f'Total events: {len(df)}, Up: {df["target"].sum()}, Down: {(1-df["target"]).sum()}')

## 2. Compute M2 Sentiment Features

For each earnings event, we compute 4 sentiment features from two data sources:

**From `window_articles` (per-article polarity in [ED-7, ED-1]):**

| Feature | Calculation | Meaning |
|---------|-------------|---------|
| `sent_mean` | Average of all article `polarity` scores in the 7-day window | Overall market tone toward this company before earnings. Higher = more positive coverage. |
| `sent_trend` | (avg polarity of later half of days) - (avg polarity of earlier half) | Is sentiment improving or deteriorating as earnings approach? Positive = improving mood. |
| `news_volume` | Total number of articles in the 7-day window | Attention/hype proxy. More articles = more market attention before earnings. |

**From `daily_sentiment` (aggregated daily score for quiet period [ED-37, ED-30]):**

| Feature | Calculation | Meaning |
|---------|-------------|---------|
| `sent_delta` | `sent_mean` - avg `sentiment` over [ED-37, ED-30] | Sentiment anomaly — how much has sentiment shifted from "normal" levels. Positive = sentiment is unusually high before this earnings event compared to a month earlier. |

The quiet period [ED-37, ED-30] serves as a baseline for what "normal" sentiment looks like for this company, far enough from earnings to be unaffected by pre-earnings anticipation.

In [ ]:
articles = pd.read_sql('SELECT * FROM window_articles', conn)
articles['article_date'] = pd.to_datetime(articles['article_date'])
articles['earnings_date'] = pd.to_datetime(articles['earnings_date'])

daily_sent = pd.read_sql('SELECT * FROM daily_sentiment', conn)
daily_sent['date'] = pd.to_datetime(daily_sent['date'])

sent_features = []

for _, row in df.iterrows():
    tk = row['ticker']
    ed = row['earnings_date']
    
    # ── Window articles [ED-7, ED-1] ──
    mask = (articles['ticker'] == tk) & (articles['earnings_date'] == ed)
    window = articles[mask].copy()
    
    if len(window) < 3:
        sent_features.append({
            'ticker': tk, 'earnings_date': ed,
            'sent_mean': np.nan, 'sent_trend': np.nan,
            'sent_delta': np.nan, 'news_volume': len(window),
        })
        continue
    
    # sent_mean: average polarity across all articles
    sent_mean = window['polarity'].mean()
    
    # sent_trend: aggregate by day, then compare later half vs earlier half
    daily_agg = window.groupby('article_date')['polarity'].mean().sort_index()
    n_days = len(daily_agg)
    if n_days >= 2:
        mid = n_days // 2
        early = daily_agg.iloc[:mid].mean()
        late = daily_agg.iloc[mid:].mean()
        sent_trend = late - early
    else:
        sent_trend = 0.0
    
    # news_volume: total articles
    news_volume = len(window)
    
    # sent_delta: sent_mean - quiet period mean
    # Quiet period: [ED-37, ED-30] from daily_sentiment
    quiet_start = ed - pd.Timedelta(days=37)
    quiet_end = ed - pd.Timedelta(days=30)
    quiet = daily_sent[
        (daily_sent['ticker'] == tk) &
        (daily_sent['date'] >= quiet_start) &
        (daily_sent['date'] <= quiet_end)
    ]
    
    if len(quiet) >= 3:
        quiet_mean = quiet['sentiment'].mean()
        sent_delta = sent_mean - quiet_mean
    else:
        sent_delta = np.nan
    
    sent_features.append({
        'ticker': tk, 'earnings_date': ed,
        'sent_mean': sent_mean,
        'sent_trend': sent_trend,
        'sent_delta': sent_delta,
        'news_volume': news_volume,
    })

sent_df = pd.DataFrame(sent_features)
df = df.merge(sent_df, on=['ticker', 'earnings_date'])

print(f'Features computed for {len(df)} events')
print(f'\nNaN counts:')
print(df[['sent_mean', 'sent_trend', 'sent_delta', 'news_volume']].isna().sum())
print(f'\nFeature summary:')
df[['surprise_pct', 'sent_mean', 'sent_trend', 'sent_delta', 'news_volume']].describe().round(3)

## 3. Feature Exploration

In [ ]:
feat_cols = ['surprise_pct', 'sent_mean', 'sent_trend', 'sent_delta', 'news_volume']

fig, axes = plt.subplots(1, len(feat_cols), figsize=(18, 4))
fig.suptitle('Feature Distributions by Target (Up=green, Down=red)', fontsize=14)

for ax, col in zip(axes, feat_cols):
    valid = df.dropna(subset=[col])
    up = valid[valid['target'] == 1][col]
    down = valid[valid['target'] == 0][col]
    ax.hist(up, bins=15, alpha=0.6, color='green', label='Up', density=True)
    ax.hist(down, bins=15, alpha=0.6, color='red', label='Down', density=True)
    ax.set_title(col, fontsize=11)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Walk-Forward Evaluation: M0 vs M1 vs M2

**Important note on comparability:** M2 requires sentiment features, which are unavailable for 2-3 early META events (EODHD has no META sentiment data before 2021-11). These events are dropped before evaluation. To ensure a fair apples-to-apples comparison, **all three models (M0, M1, M2) are evaluated on the same reduced test set**. This is why M0/M1 numbers here differ slightly from notebook 01, which used the full 91-event dataset.

In [ ]:
# Drop rows where M2 features are NaN (META early events)
df_m2 = df.dropna(subset=['sent_mean', 'sent_trend', 'sent_delta', 'news_volume']).reset_index(drop=True)
print(f'Events usable for M2: {len(df_m2)} (dropped {len(df) - len(df_m2)} with NaN)')

MIN_TRAIN = 40
y = df_m2['target'].values

# Feature sets
X_m1 = df_m2[['surprise_pct']].values
X_m2 = df_m2[['surprise_pct', 'sent_mean', 'sent_trend', 'sent_delta', 'news_volume']].values

# M0 predictions (rule-based, no training)
m0_pred_all = (df_m2['surprise_pct'].values > 0).astype(int)

results = {'M0': {'true': [], 'pred': [], 'prob': []},
           'M1': {'true': [], 'pred': [], 'prob': []},
           'M2': {'true': [], 'pred': [], 'prob': []}}

for t in range(MIN_TRAIN, len(df_m2)):
    y_test = y[t]
    
    # M0
    results['M0']['true'].append(y_test)
    results['M0']['pred'].append(m0_pred_all[t])
    results['M0']['prob'].append(float(m0_pred_all[t]))
    
    # M1
    m1 = LogisticRegression(random_state=42, max_iter=1000)
    m1.fit(X_m1[:t], y[:t])
    results['M1']['true'].append(y_test)
    results['M1']['pred'].append(m1.predict(X_m1[t:t+1])[0])
    results['M1']['prob'].append(m1.predict_proba(X_m1[t:t+1])[0, 1])
    
    # M2 (with scaling)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_m2[:t])
    X_test_scaled = scaler.transform(X_m2[t:t+1])
    m2 = LogisticRegression(random_state=42, max_iter=1000)
    m2.fit(X_train_scaled, y[:t])
    results['M2']['true'].append(y_test)
    results['M2']['pred'].append(m2.predict(X_test_scaled)[0])
    results['M2']['prob'].append(m2.predict_proba(X_test_scaled)[0, 1])

# Convert to arrays
for m in results:
    for k in results[m]:
        results[m][k] = np.array(results[m][k])

print(f'Walk-forward predictions: {len(results["M0"]["true"])} events')

## 5. Metrics Comparison

In [ ]:
metrics = {}
for name in ['M0', 'M1', 'M2']:
    yt = results[name]['true']
    yp = results[name]['pred']
    yprob = results[name]['prob']
    metrics[name] = {
        'Accuracy': accuracy_score(yt, yp),
        'F1': f1_score(yt, yp),
        'AUC': roc_auc_score(yt, yprob),
    }

metrics_df = pd.DataFrame(metrics).T
print('Model Comparison:')
print(metrics_df.round(3).to_string())

print(f'\n--- M2 Classification Report ---')
print(classification_report(results['M2']['true'], results['M2']['pred'],
                            target_names=['Down', 'Up'], zero_division=0))

## 6. M2 Model Coefficients

In [ ]:
# Final model on all data for interpretation
scaler_final = StandardScaler()
X_m2_scaled = scaler_final.fit_transform(X_m2)
m2_final = LogisticRegression(random_state=42, max_iter=1000)
m2_final.fit(X_m2_scaled, y)

feat_names = ['surprise_pct', 'sent_mean', 'sent_trend', 'sent_delta', 'news_volume']
coef_df = pd.DataFrame({
    'Feature': feat_names,
    'Coefficient': m2_final.coef_[0],
    'Abs': np.abs(m2_final.coef_[0]),
}).sort_values('Abs', ascending=False)

print(f'Intercept: {m2_final.intercept_[0]:.4f}')
print(f'\nCoefficients (standardized features):')
print(coef_df[['Feature', 'Coefficient']].to_string(index=False))
print(f'\nInterpretation: positive coef → feature increases P(Up)')

## 7. Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('M0 vs M1 vs M2: Progressive Model Comparison', fontsize=15, fontweight='bold')

# ── 1. ROC Curves ──
ax = axes[0, 0]
colors_map = {'M0': '#F44336', 'M1': '#2196F3', 'M2': '#4CAF50'}
for name in ['M1', 'M2']:
    fpr, tpr, _ = roc_curve(results[name]['true'], results[name]['prob'])
    ax.plot(fpr, tpr, color=colors_map[name], linewidth=2,
            label=f'{name} (AUC={metrics[name]["AUC"]:.3f})')
# M0 as point
cm0 = confusion_matrix(results['M0']['true'], results['M0']['pred'])
m0_fpr = cm0[0,1] / (cm0[0,0] + cm0[0,1]) if (cm0[0,0] + cm0[0,1]) > 0 else 0
m0_tpr = cm0[1,1] / (cm0[1,0] + cm0[1,1]) if (cm0[1,0] + cm0[1,1]) > 0 else 0
ax.plot(m0_fpr, m0_tpr, 'rs', markersize=12, label=f'M0 (AUC={metrics["M0"]["AUC"]:.3f})')
ax.plot([0,1],[0,1],'k--',alpha=0.4)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves'); ax.legend(loc='lower right'); ax.grid(alpha=0.3)

# ── 2. Metrics Bar Chart ──
ax = axes[0, 1]
x_pos = np.arange(3)
width = 0.25
for i, name in enumerate(['M0', 'M1', 'M2']):
    vals = [metrics[name]['Accuracy'], metrics[name]['F1'], metrics[name]['AUC']]
    bars = ax.bar(x_pos + i*width, vals, width, label=name, color=colors_map[name], alpha=0.8)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', fontsize=8, fontweight='bold')
ax.set_xticks(x_pos + width)
ax.set_xticklabels(['Accuracy', 'F1', 'AUC'])
ax.set_ylim(0, 1)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.4)
ax.set_title('M0 vs M1 vs M2 Metrics'); ax.legend(); ax.grid(axis='y', alpha=0.3)

# ── 3. Feature Importance ──
ax = axes[1, 0]
coef_sorted = coef_df.sort_values('Coefficient')
colors_bar = ['#4CAF50' if c > 0 else '#F44336' for c in coef_sorted['Coefficient']]
ax.barh(coef_sorted['Feature'], coef_sorted['Coefficient'], color=colors_bar, alpha=0.8)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_title('M2 Feature Coefficients (standardized)')
ax.set_xlabel('Coefficient')
ax.grid(alpha=0.3)

# ── 4. Cumulative Trading Returns ──
ax = axes[1, 1]
test_df = df_m2.iloc[MIN_TRAIN:].copy()
for name in ['M0', 'M1', 'M2']:
    strat_ret = np.where(results[name]['pred'] == 1, test_df['ret_5d'].values, -test_df['ret_5d'].values)
    cum_ret = (1 + strat_ret).cumprod()
    gross = (cum_ret[-1] - 1) * 100
    ax.plot(range(len(cum_ret)), cum_ret, color=colors_map[name], linewidth=2,
            label=f'{name} ({gross:+.1f}%)')
# Buy & Hold
bh = (1 + test_df['ret_5d'].values).cumprod()
bh_gross = (bh[-1] - 1) * 100
ax.plot(range(len(bh)), bh, 'gray', linewidth=1, alpha=0.5, label=f'Buy & Hold ({bh_gross:+.1f}%)')
ax.axhline(y=1.0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Event #'); ax.set_ylabel('Cumulative Return')
ax.set_title('Simulated Trading Returns'); ax.legend(loc='upper left'); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/m0_m1_m2_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/m0_m1_m2_comparison.png')

## 8. Summary

| Metric | M0 (Beat/Miss) | M1 (Surprise %) | M2 (+ Sentiment) | Random |
|--------|----------------|------------------|-------------------|--------|
| Accuracy | 0.542 | 0.419 | 0.500 | 0.500 |
| F1 | 0.667 | 0.648 | 0.600 | — |
| AUC-ROC | 0.558 | 0.617 | 0.447 | 0.500 |
| Gross Return | +24.3% | +6.8% | +30.2% | — |

### Key Findings

**1. M2's AUC (0.447) is worse than random — sentiment features add noise, not signal.**

With only ~48 test events and 5 features, the logistic regression is fitting to noise in the sentiment data. The 4 new sentiment features don't have a reliable linear relationship with the target in this small sample. This is a classic overfitting-on-small-data problem.

**2. Yet M2 has the best trading returns (+30.2%) — a paradox explained.**

AUC measures probability ranking quality, but trading returns depend on *which* events the model gets right. M2 happens to correctly call a few high-magnitude events (large moves), which boosts cumulative returns despite poor overall discrimination. This is not robust — it would likely not replicate out of sample.

**3. `surprise_pct` remains the only feature with meaningful positive signal.**

The coefficient analysis shows `surprise_pct` has the largest positive coefficient (+0.35), confirming that beating earnings expectations predicts upward moves. All four sentiment features have negative coefficients, meaning higher pre-earnings sentiment is associated with *lower* probability of post-earnings gains. This could reflect a "buy the rumor, sell the news" dynamic — when sentiment is already very positive before earnings, the stock may have already priced in good news, leaving less upside.

**4. M0's simple rule still dominates on accuracy.**

The naive "beat → up, miss → down" rule (M0) achieves the highest accuracy (0.542) because it makes actual binary decisions. M1 and M2 both struggle with the decision boundary — M1 predicts all-up, and M2's added features don't help it find a better cutoff.

### Implications for M3

The negative sentiment coefficients are actually an interesting finding — they suggest a contrarian signal that may become more powerful when combined with cross-company spillover information in M3. If company A's sentiment is high but its M7 peers' sentiment is even higher (system-wide hype), that could be a stronger sell signal than company-level sentiment alone.

The core challenge remains **sample size** (91 events, ~48 in test). M3 should be designed with this constraint in mind — adding more features risks further overfitting. The spillover network features should replace or compress the raw sentiment features rather than simply adding on top.